# Thử lịch sử 1 phút KBS

Kiểm tra FPT vào 18/11/2016 và 16/11/2026, kèm cửa sổ nhiều ngày để kiểm tra cách API xử lý hai mốc bằng nhau. Bảng coverage ghi số dòng API trả và số dòng có timestamp nằm trong khoảng yêu cầu. Dữ liệu mẫu sử dụng đơn vị giá nguồn, nằm trong RAM.


In [1]:
from pathlib import Path
import sys
import time

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
root = next(p for p in (cwd, *cwd.parents) if (p / "ingest/market/kbs/adapter.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from ingest.market.kbs.adapter import probe_minute_history


In [2]:
windows = [
    ("2016-11-18", "2016-11-18"),
    ("2026-11-16", "2026-11-16"),
    ("2016-11-14", "2016-11-18"),
    ("2026-11-09", "2026-11-16"),
]

records, frames = [], []
for start, end in windows:
    record, bars = probe_minute_history("FPT", start, end)
    records.append(record)
    if not bars.empty:
        frames.append(bars)
    time.sleep(0.5)

coverage = pd.DataFrame(records)
df = pd.concat(frames, ignore_index=True).drop_duplicates() if frames else bars
display(coverage)
display(df.head(10))


,symbol,interval,start,end,http_status,status,returned_rows,matched_rows,first_time,last_time
0,FPT,1m,2016-11-18,2016-11-18,400,BLOCKED,None,None,None,None
1,FPT,1m,2026-11-16,2026-11-16,400,BLOCKED,None,None,None,None
2,FPT,1m,2016-11-14,2016-11-18,400,BLOCKED,None,None,None,None
3,FPT,1m,2026-11-09,2026-11-16,400,BLOCKED,None,None,None,None


,time,open,high,low,close,volume,symbol,source,interval


## Đối chiếu các nguồn 1m khác

| Repo | Nguồn giá phút | Kết quả kiểm tra |
|---|---|---|
| [vnstock-js](https://github.com/ttqteo/vnstock-js) | Vietcap | Đọc mã tại commit `238d560`. Gọi trực tiếp endpoint mà adapter sử dụng cho FPT vào các tuần mẫu năm 2016, 2021 và 2026. Ba phản hồi đều HTTP 400 dạng HTML, chưa xác định được độ sâu lịch sử. |
| [quant-agent](https://github.com/tomtranai/quant-agent) | DNSE Entrade | README ghi khoảng 90 ngày cho khung nhỏ hơn 1D. Chưa xác minh độc lập cửa sổ lưu trữ của Entrade. |
| [viet-nam-stock](https://github.com/tran-minh-khoi/viet-nam-stock/blob/main/docs/API.md) | DNSE | Tài liệu xác nhận lớp API dùng khoá DNSE. Phép thử DNSE của project trên 1.000 mã có 958 mã trả nến 1m, khoảng 18/08–18/11/2026. |

Chưa xác nhận nến cổ phiếu 1m năm 2016 từ các đường gọi thử. Không cộng số mã của nhiều repo nếu chúng lấy cùng nguồn.


## Kiểm tra lại các nguồn 1m

Kết quả chạy ngày 18/11/2026, đếm timestamp nằm trong khoảng ngày yêu cầu theo giờ Việt Nam.

| Nguồn | Mã | Khoảng thử | Kết quả |
|---|---|---|---|
| KBS | FPT | 18/11/2016, 16/11/2026 và hai cửa sổ nhiều ngày | 4/4 yêu cầu HTTP 400, Request Blocked |
| Vietcap (vnstock-js) | FPT | Các tuần mẫu năm 2016, 2021 và 2026 | 3/3 yêu cầu HTTP 400, HTML |
| DNSE OpenAPI (upstream của viet-nam-stock) | FPT | 14–18/11/2016 và 04–08/01/2021 | 0 nến |
| DNSE OpenAPI | FPT | 09–16/11/2026 | 1.356 nến |
| Entrade công khai (upstream của quant-agent) | FPT | Các tuần mẫu năm 2016, 2021, 2025 và tháng 6/2026 | HTTP 200, 0 nến |
| Entrade công khai | FPT | 17–21/08/2026 | 674 nến, bắt đầu 19/08 |
| Entrade công khai | FPT, PVS, QNS | 09–16/11/2026 | Lần lượt 1.356, 1.359 và 194 nến |
| Entrade công khai | PVS, QNS | PVS 14–18/11/2016, QNS 06–10/03/2017 | HTTP 200, 0 nến |

Các lỗi HTTP 400 chưa cho biết độ sâu dữ liệu KBS/Vietcap. Các cửa sổ rỗng là kết quả mẫu, không phải kiểm kê toàn bộ kho. Lượt này chưa phát hiện nguồn bổ sung nến 1m ở mốc 2016.

Đối chiếu mã nguồn: [quant-agent 4ba8ec8](https://github.com/tomtranai/quant-agent/blob/4ba8ec8/quant_agent/technical.py), [viet-nam-stock 739ec28](https://github.com/tran-minh-khoi/viet-nam-stock/blob/739ec28/sources.py), [vnstock-js 238d560](https://github.com/ttqteo/vnstock-js/blob/238d560/src/adapters/vci.ts).
